In [1]:
import pandas as pd
import numpy as np
import time
import datetime
import json
from pybit.unified_trading import WebSocket
import websocket
from threading import Event
import traceback
import sys

In [2]:
logging_dir = "/home/kp_info_loader/"
with open("/home/kp_info_loader/kp_info_loader_config.json") as f:
    config = json.load(f)
proc_n = 5
# node = config['node']
node = 'kp_info_loader'
monitor_bot_name = config['monitor_setting']['monitor_bot']
monitor_bot_token = config['telegram_bot_setting'][monitor_bot_name]
monitor_bot_api_url = config['monitor_setting']['monitor_bot_api_url']
admin_id = config['telegram_admin_id']['charlie1155']
sys.path.append("/home/kp_info_loader")
from etc.register_monitor_msg import RegisterMonitorMsg
register_monitor_msg = RegisterMonitorMsg(monitor_bot_token, monitor_bot_api_url, admin_id, logging_dir)
# Read api keys
exchange_api_key_dict = config['exchange_api_key']
# Exchange market settings
enabled_markets_dict = config['enabled_markets']

In [3]:
info_dict = {}
from exchange_plugin.bybit_plug import InitBybitAdaptor

bybit_adaptor = InitBybitAdaptor()
info_dict['bybit_spot_info_df'] = bybit_adaptor.spot_exchange_info()
info_dict['bybit_usd_m_info_df'] = bybit_adaptor.usd_m_exchange_info()
info_dict['bybit_coin_m_info_df'] = bybit_adaptor.coin_m_exchange_info()

2023-11-09 15:02:42,183 - bybit_plug - INFO - bybit_plug_logger started.


In [4]:
from multiprocessing import Process, Manager, Event
from threading import Thread
import pandas as pd
import traceback
from pybit.unified_trading import WebSocket
import json
import datetime
import time
# set directory to upper directory
import os
import sys
# sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname(__file__)))) # Orig
sys.path.append("/home/kp_info_loader") # TEMP
from loggers.logger import KimpBotLogger
from exchange_websocket.utils import list_slice

class BybitWebsocket:
    def __init__(self, admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, info_dict, logging_dir=None):
        self.market_type = market_type
        self.admin_id = admin_id
        self.node = node
        self.register_monitor_msg = register_monitor_msg
        self.get_symbol_list = get_symbol_list
        self.info_dict = info_dict
        self.websocket_logger = KimpBotLogger(f"bybit_{self.market_type.lower()}_websocket", logging_dir).logger
        manager = Manager()
        self.ticker_dict = manager.dict()
        self.orderbook_dict = manager.dict()
        self.proc_n = proc_n
        self.before_symbols_list = self.get_symbol_list()
        self.sliced_symbols_list = list_slice(self.get_symbol_list(), self.proc_n)
        self.stop_restart_webscoket = False
        self.price_proc_event_list = []
        self.websocket_proc_dict = {}
        self.websocket_symbol_dict = {}
        self._start_websocket()
        self.monitor_shared_symbol_change_thread = Thread(target=self.monitor_shared_symbol_change, daemon=True)
        self.monitor_shared_symbol_change_thread.start()
        self.monitor_websocket_last_update_thread = Thread(target=self.monitor_websocket_last_update, daemon=True)
        self.monitor_websocket_last_update_thread.start()

    def init_ticker_websocket(self, ticker_dict, symbol_list, error_event):
        def cut_list(list):
            return [list[i:i + 10] for i in range(0, len(list), 10)]
        if self.market_type == "SPOT":
            channel_type = 'spot'
        elif self.market_type == "USD_M":
            channel_type = "linear"
        elif self.market_type == "COIN_M":
            channel_type = "inverse"
        else:
            raise Exception(f"market_type {self.market_type} is not supported.")

        ws = WebSocket(
            testnet=False,
            channel_type=channel_type
        )

        def handle_message(message):
            if error_event.is_set():
                ws.exit()
                return
            if 'data' in message.keys():
                ticker_dict[message['data']['symbol']] = {**message['data'], 'last_update': datetime.datetime.utcnow()}
        
        if self.market_type == "SPOT":
            for symbol_bunch in cut_list(symbol_list):
                ws.ticker_stream(
                    symbol=symbol_bunch,
                    callback=handle_message
                )
                time.sleep(0.1)
        else:
            ws.ticker_stream(
                symbol=symbol_list,
                callback=handle_message
            )
        # idle
        while error_event.is_set() is False:
            time.sleep(1)

    def init_orderbook_websocket(self, orderbook_dict, symbol_list, error_event):
        def cut_list(list):
            return [list[i:i + 10] for i in range(0, len(list), 10)]
        if self.market_type == "SPOT":
            channel_type = 'spot'
        elif self.market_type == "USD_M":
            channel_type = "linear"
        elif self.market_type == "COIN_M":
            channel_type = "inverse"
        else:
            raise Exception(f"market_type {self.market_type} is not supported.")

        ws = WebSocket(
            testnet=False,
            channel_type=channel_type
        )

        def handle_message(message):
            if error_event.is_set():
                ws.exit()
                return
            if 'data' in message.keys():
                orderbook_dict[message['data']['s']] = {**message['data'], 'last_update': datetime.datetime.utcnow()}

        if self.market_type == "SPOT":
            for symbol_bunch in cut_list(symbol_list):
                ws.orderbook_stream(
                    depth=1,
                    symbol=symbol_bunch,
                    callback=handle_message
                )
                time.sleep(0.1)
        else:
            ws.orderbook_stream(
                depth=1,
                symbol=symbol_list,
                callback=handle_message
            )
        # idle
        while error_event.is_set() is False:
            time.sleep(1)
    
    def _start_websocket(self):
        def handle_price_procs():
            while True:
                try:
                    if self.stop_restart_webscoket is False:
                        for i in range(self.proc_n):
                            ticker_start_proc = False
                            ticker_restarted = False
                            if f"{i+1}th_ticker_proc" in self.websocket_proc_dict.keys() and not self.websocket_proc_dict[f"{i+1}th_ticker_proc"].is_alive():
                                ticker_start_proc = True
                                ticker_restarted = True
                                self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
                                self.websocket_logger.info(f"bybit_ticker_websocket|{i+1}th bybit_ticker_proc terminated.")
                            elif f"{i+1}th_ticker_proc" not in self.websocket_proc_dict.keys():
                                ticker_start_proc = True
                                self.websocket_logger.info(f"{i+1}th Bybit ticker websocket does not exist. starting..")
                                self.websocket_logger.info(f"bybit_ticker_websocket|{i+1}th bybit_ticker_proc started.")
                            if ticker_start_proc is True:
                                error_event = Event()
                                self.price_proc_event_list.append(error_event)
                                self.websocket_symbol_dict[f"{i+1}th_ticker_symbol"] = self.sliced_symbols_list[i]
                                ticker_proc = Process(target=self.init_ticker_websocket, args=(self.ticker_dict, self.sliced_symbols_list[i], error_event), daemon=True)
                                self.websocket_proc_dict[f"{i+1}th_ticker_proc"] = ticker_proc
                                ticker_proc.start()
                                if ticker_restarted:
                                    content = f"restarted {i+1}th ticker websocket.. alive state: {self.websocket_proc_dict[f'{i+1}th_ticker_proc'].is_alive()}"
                                    self.websocket_logger.info(f"ticker_websocket|{content}")
                                    self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', f'BYBIT {self.market_type} ticker websocket restart', content, code=None, sent_switch=0, send_counts=1, remark=None)
                            time.sleep(0.5)
                            orderbook_start_proc = False
                            orderbook_restarted = False
                            if f"{i+1}th_orderbook_proc" in self.websocket_proc_dict.keys() and not self.websocket_proc_dict[f"{i+1}th_orderbook_proc"].is_alive():
                                orderbook_start_proc = True
                                orderbook_restarted = True
                                self.websocket_proc_dict[f"{i+1}th_orderbook_proc"].terminate()
                                self.websocket_logger.info(f"bybit_orderbook_websocket|{i+1}th bybit_orderbook_proc terminated.")
                            elif f"{i+1}th_orderbook_proc" not in self.websocket_proc_dict.keys():
                                orderbook_start_proc = True
                                self.websocket_logger.info(f"{i+1}th Bybit orderbook websocket does not exist. starting..")
                                self.websocket_logger.info(f"bybit_orderbook_websocket|{i+1}th bybit_orderbook_proc started.")
                            if orderbook_start_proc is True:
                                error_event = Event()
                                self.price_proc_event_list.append(error_event)
                                self.websocket_symbol_dict[f"{i+1}th_orderbook_symbol"] = self.sliced_symbols_list[i]
                                orderbook_proc = Process(target=self.init_orderbook_websocket, args=(self.orderbook_dict, self.sliced_symbols_list[i], error_event), daemon=True)
                                self.websocket_proc_dict[f"{i+1}th_orderbook_proc"] = orderbook_proc
                                orderbook_proc.start()
                                if orderbook_restarted:
                                    content = f"restarted {i+1}th orderbook websocket.. alive state: {self.websocket_proc_dict[f'{i+1}th_orderbook_proc'].is_alive()}"
                                    self.websocket_logger.info(f"orderbook_websocket|{content}")
                                    self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', f'BYBIT {self.market_type} orderbook websocket restart', content, code=None, sent_switch=0, send_counts=1, remark=None)
                            time.sleep(0.5)
                except Exception as e:
                    content = f"handle_price_procs|{traceback.format_exc()}"
                    self.websocket_logger.error(content)
                    self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"[BYBIT {self.market_type}]handle_price_procs", content=content, code=None, sent_switch=0, send_counts=1, remark=None)
                    time.sleep(1)
                time.sleep(0.5)
        self.handle_price_procs_thread = Thread(target=handle_price_procs, daemon=True)
        self.handle_price_procs_thread.start()

    def terminate_websocket(self):
        self.stop_restart_webscoket = True
        time.sleep(0.5)
        for each_event in self.price_proc_event_list:
            each_event.set()
        self.websocket_logger.info(f"[BYBIT {self.market_type}]all websockets' event has been set")
        self.price_proc_event_list = []
        time.sleep(1)
        self.stop_restart_webscoket = False
    
    def check_status(self, print_result=False, include_text=False):
        if len(self.websocket_proc_dict) == 0:
            proc_status = False
            print_text = f"[BYBIT {self.market_type}]websocket proc is not running."
            if print_result:
                print(print_text)
            if include_text:
                return (proc_status, print_text)
            return proc_status
        else:
            proc_status = all([x.is_alive() for x in self.websocket_proc_dict.values()])
            print_text = ""
            for key, value in self.websocket_proc_dict.items():
                print_text += f"[BYBIT {self.market_type}]{key} status: {value.is_alive()}\n"
            if print_result:
                print(print_text)
            if include_text:
                return (proc_status, print_text)
            return proc_status

    def monitor_shared_symbol_change(self, loop_time_secs=60):
        self.websocket_logger.info(f"[BYBIT {self.market_type}]started monitor_shared_symbol_change..")
        while True:
            time.sleep(loop_time_secs)
            try:
                restart_websockets = False
                new_symbols_list = self.get_symbol_list()
                
                if sorted(self.before_symbols_list) != sorted(new_symbols_list):
                    restart_websockets = True
                    deleted_spot_shared_symbol = [x for x in self.before_symbols_list if x not in new_symbols_list]
                    added_spot_shared_symbol = [x for x in new_symbols_list if x not in self.before_symbols_list]
                    content = f"monitor_shared_symbol_change|[BYBIT {self.market_type}]shared symbol changed. deleted: {deleted_spot_shared_symbol}, added: {added_spot_shared_symbol}"
                    self.websocket_logger.info(content)
                    self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_shared_symbol_change', content, code=None, sent_switch=0, send_counts=1, remark=None)
                    for each_shared_symbol in deleted_spot_shared_symbol:
                        # remove deleted symbol from ticker_dict
                        try:
                            del self.ticker_dict[each_shared_symbol]
                        except Exception:
                            pass
                
                if restart_websockets is True:
                    # Set the newer values to before values
                    self.before_symbols_list = new_symbols_list
                    # Set sliced values too
                    self.sliced_symbols_list = list_slice(self.get_symbol_list(), self.proc_n)
                    # terminate websockets
                    self.terminate_websocket()
            except Exception as e:
                content = f"monitor_shared_symbol_change|{traceback.format_exc()}"
                self.websocket_logger.error(content)
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"monitor_shared_symbol_change", content=content, code=None, sent_switch=0, send_counts=1, remark=None)

    def monitor_websocket_last_update(self, update_threshold_mins=10, loop_time_secs=15):
        self.websocket_logger.info(f"[BYBIT {self.market_type}]started monitor_websocket_last_update..")
        while True:
            time.sleep(loop_time_secs)
            try:
                ticker_df = pd.DataFrame(dict(self.ticker_dict)).T.reset_index()
                for i in range(self.proc_n):
                    allocated_symbol_list = self.websocket_symbol_dict[f"{i+1}th_ticker_symbol"]
                    # check ticker dict's last_update
                    allocated_ticker_df = ticker_df[ticker_df['symbol'].isin(allocated_symbol_list)]
                    if len(allocated_ticker_df) == 0:
                        content = f"monitor_websocket_last_update|{i+1}th_ticker_proc has no ticker_dict data. Restarting websocket.."
                        self.websocket_logger.info(content)
                        self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_websocket_last_update', content, code=None, sent_switch=0, send_counts=1, remark=None)
                        self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
                        continue
                    ticker_last_update = allocated_ticker_df['last_update'].max()
                    # check orderbook dict's last_update
                    # If the last update is older than update_threshold_mins, restart websocket
                    if (datetime.datetime.utcnow() - ticker_last_update).total_seconds() / 60 > update_threshold_mins:
                        content = f"monitor_websocket_last_update|{i+1}th_ticker_proc last_update is older than {update_threshold_mins} mins. Restarting websocket.."
                        self.websocket_logger.info(content)
                        self.register_monitor_msg.register(self.admin_id, self.node, 'monitor', 'monitor_websocket_last_update', content, code=None, sent_switch=0, send_counts=1, remark=None)
                        self.websocket_proc_dict[f"{i+1}th_ticker_proc"].terminate()
            except Exception as e:
                content = f"monitor_websocket_last_update|{traceback.format_exc()}"
                self.websocket_logger.error(content)
                self.register_monitor_msg.register(self.admin_id, self.node, 'error', f"[BYBIT {self.market_type}] monitor_websocket_last_update", content=content, code=None, sent_switch=0, send_counts=1, remark=None)

    def get_price_df(self):
        ticker_df = pd.DataFrame(self.ticker_dict.values())
        orderbook_df = pd.DataFrame(self.orderbook_dict.values())
        merged_df = ticker_df.merge(orderbook_df, left_on='symbol', right_on='s', how='inner')
        merged_df = merged_df.merge(self.info_dict[f'bybit_{self.market_type.lower()}_info_df'][['symbol','base_asset','quote_asset']], on='symbol', how='inner')
        merged_df.loc[:, 'b'] = merged_df['b'].apply(lambda x: x[0][0])
        merged_df.loc[:, 'a'] = merged_df['a'].apply(lambda x: x[0][0])
        merged_df.loc[:, 'price24hPcnt'] = merged_df['price24hPcnt'].astype(float) * 100
        if self.market_type == "COIN_M":
            merged_df = merged_df.rename(columns={"lastPrice":'tp', 'a':'ap', 'b':'bp', 'price24hPcnt':'scr', 'volume24h':'atp24h'})
        else:
            merged_df = merged_df.rename(columns={"lastPrice":'tp', 'a':'ap', 'b':'bp', 'price24hPcnt':'scr', 'turnover24h':'atp24h'})
        merged_df.loc[:, ['tp','ap','bp','scr','atp24h']] = merged_df[['tp','ap','bp','scr','atp24h']].astype(float)
        merged_df = merged_df[['symbol','base_asset','quote_asset','tp','bp','ap','scr','atp24h']]
        return merged_df

class BybitUSDMWebsocket(BybitWebsocket):
    def __init__(self, admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, info_dict, logging_dir):
        super().__init__(admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, info_dict, logging_dir)

class BybitCOINMWebsocket(BybitWebsocket):
    def __init__(self, admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, info_dict, logging_dir):
        super().__init__(admin_id, node, proc_n, get_symbol_list, register_monitor_msg, market_type, info_dict, logging_dir)
        


In [5]:
bybit_websocket = BybitWebsocket(admin_id, node, 1, lambda : ["BTCUSDT","ETHUSDT"], register_monitor_msg, "SPOT", info_dict, logging_dir=None)

2023-11-09 15:02:45,521 - bybit_spot_websocket - INFO - 1th Bybit ticker websocket does not exist. starting..


2023-11-09 15:02:45,537 - bybit_spot_websocket - INFO - bybit_ticker_websocket|1th bybit_ticker_proc started.
2023-11-09 15:02:46,053 - bybit_spot_websocket - INFO - 1th Bybit orderbook websocket does not exist. starting..
2023-11-09 15:02:46,063 - bybit_spot_websocket - INFO - bybit_orderbook_websocket|1th bybit_orderbook_proc started.


In [6]:
bybit_websocket.get_price_df()

,symbol,base_asset,quote_asset,tp,bp,ap,scr,atp24h
0,BTCUSDT,BTC,USDT,36609.99,36609.99,36610.0,3.89,446594640.872574
1,ETHUSDT,ETH,USDT,1919.2,1919.19,1919.2,2.22,156770895.455298


In [7]:
bybit_usd_m_websocket = BybitUSDMWebsocket(admin_id, node, 1, lambda : ["BTCUSDT","ETHUSDT"], register_monitor_msg, "USD_M", info_dict, logging_dir=None)

2023-11-09 15:02:51,359 - bybit_usd_m_websocket - INFO - 1th Bybit ticker websocket does not exist. starting..


2023-11-09 15:02:51,363 - bybit_usd_m_websocket - INFO - bybit_ticker_websocket|1th bybit_ticker_proc started.


2023-11-09 15:02:51,894 - bybit_usd_m_websocket - INFO - 1th Bybit orderbook websocket does not exist. starting..
2023-11-09 15:02:51,897 - bybit_usd_m_websocket - INFO - bybit_orderbook_websocket|1th bybit_orderbook_proc started.


In [8]:
bybit_usd_m_websocket.get_price_df()

,symbol,base_asset,quote_asset,tp,bp,ap,scr,atp24h
0,BTCUSDT,BTC,USDT,36639.0,36638.9,36639.0,3.8829,7275497700.3875
1,ETHUSDT,ETH,USDT,1920.31,1920.3,1920.31,2.2676,1334420907.5123


In [9]:
bybit_coin_m_websocket = BybitCOINMWebsocket(admin_id, node, 1, lambda : ["BTCUSD","ETHUSD"], register_monitor_msg, "COIN_M", info_dict, logging_dir=None)

2023-11-09 15:02:55,667 - bybit_coin_m_websocket - INFO - 1th Bybit ticker websocket does not exist. starting..
2023-11-09 15:02:55,671 - bybit_coin_m_websocket - INFO - bybit_ticker_websocket|1th bybit_ticker_proc started.


2023-11-09 15:02:56,183 - bybit_coin_m_websocket - INFO - 1th Bybit orderbook websocket does not exist. starting..
2023-11-09 15:02:56,185 - bybit_coin_m_websocket - INFO - bybit_orderbook_websocket|1th bybit_orderbook_proc started.


In [11]:
bybit_coin_m_websocket.get_price_df()

,symbol,base_asset,quote_asset,tp,bp,ap,scr,atp24h
0,BTCUSD,BTC,USD,36653.5,36653.0,36653.5,3.9049,939409504.0
1,ETHUSD,ETH,USD,1920.15,1920.85,1920.9,2.2253,109169204.0
